<a href="https://colab.research.google.com/github/loukikjoshi06-ops/NIRVANA-gpt/blob/main/NIRVANA_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F


here dataset is uploaded

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
with open('TinyStories-small.txt','r', encoding='utf-8')as l:
    text = l.read()

In [ ]:
print(text[ :100])

In [ ]:
char = sorted(list(set(text)))
print (len(char))

In [ ]:
stoi = {ch:i for i,ch in enumerate(char)}
itos = {i:ch for i,ch in enumerate(char)}


In [ ]:
encode = lambda s:[stoi[c] for c in s]
decode = lambda l:''.join([itos[i] for i in l])

In [ ]:
test = "loukik"
print(encode(test))
print(decode(encode(test)))

In [ ]:
data =torch.tensor(encode(text),dtype = torch.long)

In [ ]:
print(data.shape)
print(data[:50])

In [ ]:
n = int(0.8 * len(data))
train_data = data[ :n]
val_data = data [n: ]

In [ ]:
batch_size = 32
block_size = 64

In [ ]:
def get_batch(split):
  data = train_data if split == 'train' else val_data

  # Ensure 'size' is a tuple by adding a comma: (batch_size,)
  ix = torch.randint(len(data) - block_size , (batch_size,))

  x = torch.stack([data[ i : i+block_size] for i in ix])
  y = torch.stack([data[ i+1 : i+block_size+1] for i in ix])

  return x,y

In [ ]:
xb,yb = get_batch('train')
print(xb.shape)
print(yb.shape)

In [ ]:
for b in range(1):
  for t in range(block_size):
    context = xb[b,:t+1]
    target = yb[b,t]
    print(context.tolist(),"-->",target.item())

In [ ]:
class BigramLanguageModel (nn.Module):
  def __init__(self):
   super().__init__()
   vocab_size = len(char) # Define vocab_size using the global 'char' variable
   self.token_embedding_table = nn.Embedding (vocab_size,vocab_size)

  def forward(self,idx,target = None):
    logits = self.token_embedding_table(idx)

    if target == None:
      loss = None
      return logits, loss # Explicitly return logits and None for loss

    else:
      B,T,C = logits.shape
      logits = logits.view(B*T,C)
      targets = target.view(B*T)

      loss = F.cross_entropy(logits,targets)
      return logits,loss

  def generate(self,idx,max_new_tokens):
    for _ in range(max_new_tokens):
      logits,loss = self(idx)

      logits = logits[:,-1,:]

      probs = F.softmax(logits,dim=-1)

      idx_next = torch.multinomial(probs,num_samples=1)

      idx = torch.cat((idx,idx_next),dim=1)

    return idx

In [ ]:
model = BigramLanguageModel()

xb,yb = get_batch('train')

logits,loss = model(xb,yb)
print("Logit shape : ", logits.shape)
print("Loss : ", loss)

In [ ]:
context = torch.zeros((1,1),dtype = torch.long)

generated = model.generate(context,max_new_tokens = 100)

print(decode(generated[0].tolist()))


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
batch_size = 32
for iter in range(10000):
  xb,yb = get_batch('train')
  logits,loss = model(xb,yb)
  optimizer.zero_grad(set_to_none = True)
  loss.backward()
  optimizer.step()
  print(loss.item())